# Source E Draft: Medi-Cal FFS provider availability

Separate, unexecuted analysis draft; the original `source_e_analysis.ipynb` remains unchanged. The priority is to count FFS provider identities and service locations accurately, with a clear statewide and Riverside/San Bernardino reference count.

**Known source reference:** the original exploratory notebook reports 380,896 combined CSV rows and 47 columns. That is a raw row count, **not** the number of unique providers. The cells below calculate unique NPI count, provider identity count, location count, billing-provider count, and county/ZIP counts from the full local files. The CSV excerpts in chat are truncated, so do not infer those final counts from the excerpts.

The FFS directory does not provide enrolled-patient totals. `Medicaid_Patients` is an acceptance-status field, not a patient count. Provider locations per 1,000 members requires a separate, date-aligned Medi-Cal enrollment denominator.

## Source references

- Original exploratory notebook: `source_e_analysis.ipynb`
- FFS CSV parts 1 and 2: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/zipped_bc_too_big_for_github/`
- Dataset metadata: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/datapackage.json`
- FI provider type and specialty references: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/fi-provider-type-code-reference.csv` and `fi-provider-specialty-code-reference.csv`
- IEHP NPI-deduplicated comparison file: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/07_UNIQUE_IEHP_PROVIDERS_DEDUPED_BY_NPI.csv`

Metadata identifies the FFS listing as a point-in-time enrollment/geographic source dated October 26, 2026. Confirm date alignment and county definitions before comparing it with IEHP member counts.

## Counting definitions

- **Raw rows:** source CSV records; repeated specialty or service-location rows remain included.
- **Unique NPIs:** distinct valid 10-digit NPI values.
- **Unique provider identities:** valid NPI where available, otherwise a clearly labeled Provider_Number fallback. Report the fallback separately.
- **Provider-location records:** unique identity + owner/location identifiers + normalized service address fields. This is a directory location count, not capacity or patient volume.
- **Billing locations:** location records with at least one source row explicitly labeled `FFS Billing Provider`; non-billing and unclassified locations are reported separately.
- County/ZIP counts are location counts. A provider with multiple locations contributes once per location; totals across counties or ZIPs need not equal statewide unique providers. CountyName (coordinate-derived) is preferred when present; County is fallback. Disagreements are surfaced for review.

In [ ]:
from pathlib import Path
import pandas as pd

relative_data_dir = Path('data/provider-side/E-mediCal-fee-for-service-enrolled-providers')
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / relative_data_dir).is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate Source E data. Start Jupyter in the repository or a subfolder.')
source_dir = repo_root / relative_data_dir
partition_dir = source_dir / 'zipped_bc_too_big_for_github'
ffs_files = [
    partition_dir / 'medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv',
    partition_dir / 'medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv',
]
missing_files = [str(p) for p in ffs_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError('Missing FFS CSV part(s): ' + '; '.join(missing_files))

# Keep identifiers, ZIPs, license values, and phone numbers as text; only blank CSV fields become missing.
read_options = dict(encoding='cp1252', dtype='string', keep_default_na=False, na_values=[''])
providers_df = pd.concat([pd.read_csv(path, **read_options) for path in ffs_files], ignore_index=True)
print(f'Loaded {len(providers_df):,} source rows and {providers_df.shape[1]} columns')
print('Expected shape from original notebook inspection: (380,896, 47)')
if providers_df.shape != (380896, 47):
    print('NOTE: observed shape differs from the earlier snapshot; verify input files/version before comparing counts.')


## Normalize identifiers and locations

Normalization trims whitespace and standardizes address case/spacing, but does not fuzzy-match names or addresses. No rows are silently removed.

In [ ]:
def clean(col):
    return providers_df[col].astype('string').str.strip().replace('', pd.NA) if col in providers_df.columns else pd.Series(pd.NA, index=providers_df.index, dtype='string')

for col in ['NPI', 'Provider_Number', 'Owner_Number', 'Service_Location_Number', 'Legal_Name', 'Address', 'Address2', 'City', 'State', 'ZIP', 'County', 'CountyName', 'Billing_Provider']:
    providers_df[col + '_clean'] = clean(col)

providers_df['valid_npi'] = providers_df['NPI_clean'].str.fullmatch(r'\d{10}', na=False)
providers_df['provider_id'] = pd.NA
providers_df.loc[providers_df['valid_npi'], 'provider_id'] = 'NPI:' + providers_df.loc[providers_df['valid_npi'], 'NPI_clean']
fallback_mask = ~providers_df['valid_npi'] & providers_df['Provider_Number_clean'].notna()
providers_df.loc[fallback_mask, 'provider_id'] = 'PROVIDER_NUMBER:' + providers_df.loc[fallback_mask, 'Provider_Number_clean']
providers_df['id_source'] = pd.NA
providers_df.loc[providers_df['valid_npi'], 'id_source'] = 'NPI'
providers_df.loc[fallback_mask, 'id_source'] = 'Provider_Number fallback'

def norm_address(col):
    return providers_df[col + '_clean'].str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()

for col in ['Address', 'Address2', 'City', 'State']:
    providers_df[col + '_key'] = norm_address(col)
providers_df['ZIP_key'] = providers_df['ZIP_clean'].str.replace(r'\.0$', '', regex=True)
providers_df['analysis_county'] = providers_df['CountyName_clean'].combine_first(providers_df['County_clean'])
providers_df['analysis_county'] = providers_df['analysis_county'].str.strip().str.replace(r'\s+County$', '', case=False, regex=True)

print('Rows with valid 10-digit NPI:', f"{providers_df['valid_npi'].sum():,}")
print('Rows using Provider_Number fallback:', f'{fallback_mask.sum():,}')
print('Rows without a usable provider identifier:', f"{providers_df['provider_id'].isna().sum():,}")
print('Exact duplicate source rows:', f"{providers_df.duplicated().sum():,}")

county_disagreement = (providers_df['CountyName_clean'].notna() & providers_df['County_clean'].notna() &
                      providers_df['CountyName_clean'].str.replace(r'\s+County$', '', case=False, regex=True).str.casefold().ne(providers_df['County_clean'].str.casefold()))
print('Rows where CountyName and County disagree:', f'{county_disagreement.sum():,}')
display(providers_df.loc[county_disagreement, ['County_clean', 'CountyName_clean', 'ZIP_clean']].head(20))

## Build provider-location records and calculate reference counts

Specialty rows are collapsed only when provider identity, owner/service-location numbers, and normalized address fields agree. Review multi-row keys and incomplete location keys; this approach is an explicit counting rule, not a DHCS-certified unique-location identifier. Counts are calculated from the full files when this cell is run.

In [ ]:
id_rows = providers_df.loc[providers_df['provider_id'].notna()].copy()
location_components = ['provider_id', 'Owner_Number_clean', 'Service_Location_Number_clean', 'Address_key', 'Address2_key', 'City_key', 'State_key', 'ZIP_key']
id_rows['provider_location_key'] = id_rows[location_components].fillna('<MISSING>').astype('string').agg(' | '.join, axis=1)

def billing_status(values):
    vals = set(values.dropna().astype(str).str.strip())
    if 'FFS Billing Provider' in vals:
        return 'FFS Billing Provider'
    if 'FFS Non-Billing Provider' in vals:
        return 'FFS Non-Billing Provider'
    return 'Unclassified'

provider_locations = (id_rows.groupby('provider_location_key', dropna=False)
    .agg(provider_id=('provider_id', 'first'),
         NPI=('NPI_clean', 'first'),
         id_source=('id_source', 'first'),
         Provider_Number=('Provider_Number_clean', 'first'),
         analysis_county=('analysis_county', 'first'),
         ZIP=('ZIP_key', 'first'),
         State=('State_clean', 'first'),
         Legal_Name=('Legal_Name_clean', 'first'),
         billing_status=('Billing_Provider_clean', billing_status),
         source_rows=('provider_id', 'size'))
    .reset_index())

# Reference summary: each metric is explicitly labeled to avoid confusing raw rows, NPIs, identities, and locations.
reference_counts = pd.DataFrame([
    ('Combined source CSV rows', len(providers_df)),
    ('Rows with valid 10-digit NPI', int(providers_df['valid_npi'].sum())),
    ('Distinct valid NPIs', int(providers_df.loc[providers_df['valid_npi'], 'NPI_clean'].nunique())),
    ('Distinct Provider_Number fallback IDs', int(providers_df.loc[fallback_mask, 'Provider_Number_clean'].nunique())),
    ('Rows with no usable identifier', int(providers_df['provider_id'].isna().sum())),
    ('Distinct provider identities (NPI plus fallback)', int(providers_df['provider_id'].nunique())),
    ('Provider-location records (deduplicated specialty rows)', int(provider_locations['provider_location_key'].nunique())),
    ('FFS billing provider-location records', int(provider_locations['billing_status'].eq('FFS Billing Provider').sum())),
    ('FFS non-billing provider-location records', int(provider_locations['billing_status'].eq('FFS Non-Billing Provider').sum())),
    ('Unclassified provider-location records', int(provider_locations['billing_status'].eq('Unclassified').sum())),
    ('Provider-location records with California state address', int(provider_locations['State'].str.upper().eq('CA').sum())),
    ('Riverside or San Bernardino provider-location records', int(provider_locations['analysis_county'].str.casefold().isin(['riverside', 'san bernardino']).sum())),
    ('Riverside or San Bernardino distinct provider identities', int(provider_locations.loc[provider_locations['analysis_county'].str.casefold().isin(['riverside', 'san bernardino']), 'provider_id'].nunique())),
])
reference_counts.columns = ['Measure', 'Count']
display(reference_counts)
print('Reference raw row count previously reported by original notebook: 380,896')
print('Interpret the unique-provider and location rows above as computed estimates under the documented key; not as patient capacity or utilization.')

print('Provider-location keys aggregating multiple source rows:', f"{(provider_locations['source_rows'] > 1).sum():,}")
print('Provider-location records with missing county:', f"{provider_locations['analysis_county'].isna().sum():,}")
print('Provider-location records with missing ZIP:', f"{provider_locations['ZIP'].isna().sum():,}")
display(provider_locations.sort_values('source_rows', ascending=False).head(15))

## County and ZIP reference counts

These tables give the location-based counts for the requested geographies. Providers with service locations in multiple areas appear in each area. The Riverside/San Bernardino count is not the number of all IEHP providers; it is the Source E directory count assigned to those county labels.

In [ ]:
def location_summary(frame, groups):
    out = (frame.dropna(subset=groups).groupby(groups, dropna=False)
           .agg(ffs_provider_locations=('provider_location_key', 'nunique'),
                distinct_provider_identities=('provider_id', 'nunique'),
                billing_provider_locations=('billing_status', lambda s: s.eq('FFS Billing Provider').sum()),
                nonbilling_provider_locations=('billing_status', lambda s: s.eq('FFS Non-Billing Provider').sum()))
           .reset_index())
    return out.sort_values('ffs_provider_locations', ascending=False)

county_summary = location_summary(provider_locations, ['analysis_county'])
county_zip_summary = location_summary(provider_locations, ['analysis_county', 'ZIP'])
target_counties = county_summary.loc[county_summary['analysis_county'].str.casefold().isin(['riverside', 'san bernardino'])]
print('All-county results (top 20):')
display(county_summary.head(20))
print('Riverside and San Bernardino reference counts:')
display(target_counties)
print('Top ZIPs in Riverside and San Bernardino:')
display(county_zip_summary.loc[county_zip_summary['analysis_county'].str.casefold().isin(['riverside', 'san bernardino'])].head(40))

## IEHP comparison and provider-per-1,000 rate

The IEHP deduplicated CSV is a matched IEHP subset, not the complete FFS provider universe. Use it as a cross-check only. For provider locations per 1,000 patients, provide a separate member-count file aligned to the same geography, population, and reference period. The formula is `FFS provider locations / enrolled patients * 1,000`. No rate is produced without that denominator.

In [ ]:
iehp_path = source_dir / '07_UNIQUE_IEHP_PROVIDERS_DEDUPED_BY_NPI.csv'
if iehp_path.is_file():
    iehp_df = pd.read_csv(iehp_path, encoding='utf-8-sig', dtype='string', keep_default_na=False, na_values=[''])
    print(f'IEHP comparison rows: {len(iehp_df):,}')
    if 'ffs_match_classification' in iehp_df.columns:
        display(iehp_df['ffs_match_classification'].value_counts(dropna=False).to_frame('rows'))
    else:
        print('Match classification column not found; inspect headers before comparison.')
else:
    print(f'IEHP CSV is not present in this checkout: {iehp_path}')

patient_counts_path = None  # Set to a reviewed CSV with analysis_county and enrolled_patients columns
if patient_counts_path is None:
    print('No patient denominator supplied; per-1,000 rate intentionally not calculated.')
else:
    patient_counts = pd.read_csv(patient_counts_path, dtype='string', keep_default_na=False, na_values=[''])
    required = {'analysis_county', 'enrolled_patients'}
    if not required.issubset(patient_counts.columns):
        raise ValueError(f'Denominator file must contain {sorted(required)}')
    patient_counts['analysis_county'] = patient_counts['analysis_county'].astype('string').str.strip().str.replace(r'\s+County$', '', case=False, regex=True)
    patient_counts['enrolled_patients'] = pd.to_numeric(patient_counts['enrolled_patients'], errors='coerce')
    if patient_counts['analysis_county'].duplicated().any():
        raise ValueError('Provide one denominator row per county or aggregate with documented rules first.')
    rates = county_summary.merge(patient_counts, on='analysis_county', how='left', validate='one_to_one')
    rates['ffs_provider_locations_per_1000_patients'] = rates['ffs_provider_locations'] / rates['enrolled_patients'] * 1000
    display(rates.sort_values('ffs_provider_locations_per_1000_patients', ascending=False))

## Review before reporting

Run all cells and record the generated `reference_counts` output as the final source-based count, with both raw rows and deduplicated provider/location metrics clearly labeled. Verify the input snapshot, invalid/fallback IDs, incomplete locations, County/CountyName disagreements, and multi-row location keys. Report all-enrolled and billing-only figures separately. Provider counts represent listed enrollment supply—not actual appointments, capacity, patient volume, or the number accepting new patients.